In [15]:
import os
import re
import json
import glob
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt



warnings.filterwarnings("ignore")

## НАСТРОЙКИ

In [16]:
# Можно оставить True, если хочешь использовать известные точки/проявления
# как дополнительную калибровку авторской модели.
# Если хочешь полностью "слепой" прогноз без опоры на них — поставь False.
USE_EVIDENCE_GUIDANCE = True

CELL_SIZE = 500
RANDOM_STATE = 42

# Вес геологической части, локальных пересечений и калибровки по точкам
W_GEO = 0.62
W_LOCAL = 0.18
W_EVID = 0.20 if USE_EVIDENCE_GUIDANCE else 0.0

## ПУТИ

In [17]:
def find_existing_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]

    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base

    raise FileNotFoundError(
        "Не найден каталог с shp_dbf. "
        "Укажи BASE_DIR вручную или распакуй архив рядом с ноутбуком."
    )

BASE_DIR = find_existing_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "fixed_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAFE_ALIAS_DIR = OUT_DIR / "_safe_shp_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "gold_forecast_fixed.gpkg"
OUT_PNG = OUT_DIR / "gold_forecast_fixed.png"
OUT_PROX_MAGM = OUT_DIR / "prox_magm_fixed.png"
OUT_COMPARE = OUT_DIR / "compare_prox_and_final.png"
OUT_METRICS = OUT_DIR / "metrics.json"
OUT_CSV = OUT_DIR / "grid_attributes.csv"

## СЛУЖЕБНЫЕ ФУНКЦИИ

In [18]:
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr)
    mx = np.nanmax(arr)
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    return (arr - mn) / (mx - mn)

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        match = re.search(r"pj4=(.+)", txt)
        if match:
            return match.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    """
    Делает копии шейпов с битыми/не-ASCII именами в безопасные имена.
    Это нужно, чтобы geopandas стабильно читал все файлы в Linux и Windows.
    """
    aliases = {}
    stems = {}

    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")):
            continue
        if name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            base_s = None
            safe = False

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias_name = f"evidence_{alias_idx:02d}"
        alias_idx += 1

        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias_name}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)

        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias_name}_shp.pj4")

        aliases[alias_name] = alias_dir / f"{alias_name}.shp"

    return aliases

def load_layer(shp_path: Path):
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()

    if gdf.crs is None:
        proj4 = read_sidecar_proj4(shp_path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf: gpd.GeoDataFrame, target_crs):
    if gdf.crs is None and target_crs is None:
        return gdf
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask: gpd.GeoDataFrame, cell_size: int):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)

    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1

    grid = gpd.GeoDataFrame(
        rows,
        columns=["cell_id", "row", "col", "geometry"],
        geometry="geometry",
        crs=mask.crs,
    )
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid: gpd.GeoDataFrame, source: gpd.GeoDataFrame, name: str):
    source_union = unary_union(source.geometry)

    # ВАЖНО:
    # расстояние считаем от всей ячейки, а не от центроида.
    # Если ячейка пересекает объект — расстояние = 0.
    distances = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        distances[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)

    grid[name] = distances
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    """
    Сначала слегка выпрямляем распределение расстояний,
    потом переводим в proximity через экспоненциальный спад.

    Это не копия алгоритма из PDF.
    Это более аккуратная авторская схема:
    - сохраняет смысл близости,
    - не размывает фактор слишком сильно,
    - дает устойчивую шкалу между 0 и 1.
    """
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)

    if transform == "sqrt":
        dt = np.sqrt(d)
    elif transform == "cbrt":
        dt = np.cbrt(d)
    elif transform == "log1p":
        dt = np.log1p(d)
    else:
        dt = d

    scale = float(np.nanquantile(dt, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(dt)), 1.0)

    prox = np.exp(-dt / scale)
    prox = np.clip(prox, 0, 1)
    return prox

def smooth_on_regular_grid(grid: gpd.GeoDataFrame, value_col: str, shape, passes=1):
    """
    Мягкое сглаживание по соседним ячейкам 3x3.
    Нужное именно для регулярной сети, чтобы убрать рваные пятна.
    """
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()

    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()

    kernel = np.array(
        [
            [1.0, 1.5, 1.0],
            [1.5, 4.0, 1.5],
            [1.0, 1.5, 1.0],
        ],
        dtype=float,
    )

    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)

        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)

        with np.errstate(divide="ignore", invalid="ignore"):
            smoothed = np.where(den > 0, num / den, np.nan)

    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def collect_evidence_points(mask_crs, aliases):
    """
    Берем только Point/MultiPoint-слои как прямые проявления/контроль.
    Линейные аномалии просто можно использовать отдельно как контекст,
    но в target их лучше не смешивать.
    """
    point_layers = []
    for name, shp_path in aliases.items():
        if name in {
            "svita_new", "fasii", "glub_raz_nw", "glub_r_nw",
            "gr_dol_vp_poly", "kory", "dayki_buf"
        }:
            continue

        gdf = load_layer(shp_path)
        gdf = to_crs_safe(gdf, mask_crs)

        geom_types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in geom_types or "MultiPoint" in geom_types:
            point_layers.append(gdf)

    if not point_layers:
        return None

    points = pd.concat(point_layers, ignore_index=True)
    return gpd.GeoDataFrame(points, geometry="geometry", crs=mask_crs)

def plot_final_map(grid, mask, points, out_png: Path):
    """
    Визуализация сделана так, чтобы по смыслу быть ближе к примеру:
    - базовый прогностический фон: синий-белый-красный
    - лучшие зоны дополнительно выделяются желтым
    """
    fig, ax = plt.subplots(figsize=(12, 12))

    grid.plot(
        column="prognoz",
        ax=ax,
        cmap="bwr",
        linewidth=0,
        alpha=0.55,
        legend=True,
        legend_kwds={"shrink": 0.6, "label": "prognoz (меньше = перспективнее)"},
    )

    # Лучшие ~10% ячеек показываем желтым поверх фона
    grid[grid["top10"] == 1].plot(ax=ax, color="yellow", linewidth=0, alpha=0.90)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if points is not None and len(points) > 0:
        points.plot(
            ax=ax,
            color="yellow",
            markersize=14,
            edgecolor="black",
            linewidth=0.35,
        )

    ax.set_title("Исправленный итоговый прогноз", fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_prox_magm(grid, mask, out_png: Path):
    fig, ax = plt.subplots(figsize=(12, 12))
    grid.plot(
        column="prox_magm",
        ax=ax,
        cmap="RdYlBu_r",
        linewidth=0,
        legend=True,
        legend_kwds={"shrink": 0.6, "label": "prox_magm"},
    )
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    ax.set_title("Исправленный magm proximity", fontsize=14)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png: Path):
    fig, axes = plt.subplots(1, 2, figsize=(16, 9))

    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    grid.plot(column="prognoz", ax=axes[1], cmap="bwr", linewidth=0, alpha=0.55)
    grid[grid["top10"] == 1].plot(ax=axes[1], color="yellow", linewidth=0, alpha=0.90)
    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)
    if points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=12, edgecolor="black", linewidth=0.35)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

## ЗАГРУЗКА ДАННЫХ

In [19]:
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_evidence_points(mask.crs, aliases)

## СЕТКА

In [20]:
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

## ДИСТАНЦИИ

In [21]:
grid = add_distance_feature(grid, facies, "dist_facies")
grid = add_distance_feature(grid, paleo, "dist_paleo")
grid = add_distance_feature(grid, struct, "dist_struct")
grid = add_distance_feature(grid, magm, "dist_magm")
grid = add_distance_feature(grid, tect1, "dist_tect1")
grid = add_distance_feature(grid, tect2, "dist_tect2")

## PROXIMITY-ПРИЗНАКИ

In [22]:
# Важная правка:
# dayki_buf уже буферизованы, поэтому для magm нужен более "острый" спад.
# Иначе фактор размывается слишком широко.

grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], transform="cbrt", q=0.75)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], transform="cbrt", q=0.75)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], transform="sqrt", q=0.75)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], transform="sqrt", q=0.60)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], transform="cbrt", q=0.75)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], transform="cbrt", q=0.75)

# Дополнительные признаки пересечения/совпадения факторов
grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]

# Вместо простого произведения берем геометрическое среднее.
# Оно меньше переусиливает случайные совпадения.
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])
grid["facies_paleo_intersection"] = np.sqrt(grid["prox_facies"] * grid["prox_paleo"])

## АВТОРСКАЯ ГЕОЛОГИЧЕСКАЯ ОЦЕНКА

In [23]:
# Логика:
# - разломы задают основной каркас коридоров;
# - paleo/struct стабилизируют положение перспективных зон;
# - facies и magm усиливают локальные участки;
# - пересечения дают локальный бонус, но не должны доминировать.

grid["regional_score_raw"] = (
    0.24 * grid["tect_combo"] +
    0.18 * grid["prox_paleo"] +
    0.16 * grid["prox_struct"] +
    0.13 * grid["prox_facies"] +
    0.11 * grid["prox_magm"] +
    0.10 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"]
)

grid["local_score_raw"] = (
    0.40 * grid["tect_magm_intersection"] +
    0.25 * grid["tect_struct_intersection"] +
    0.20 * grid["paleo_struct_intersection"] +
    0.15 * grid["facies_paleo_intersection"]
)

grid["regional_score"] = normalize_01(grid["regional_score_raw"])
grid["local_score"] = normalize_01(grid["local_score_raw"])

# Сглаживаем только саму оценку, а не исходные признаки
grid["regional_score_sm"] = normalize_01(
    smooth_on_regular_grid(grid, "regional_score", grid_shape, passes=2)
)
grid["local_score_sm"] = normalize_01(
    smooth_on_regular_grid(grid, "local_score", grid_shape, passes=1)
)

grid["geo_score"] = normalize_01(
    0.78 * grid["regional_score_sm"] +
    0.22 * grid["local_score_sm"]
)

## ДОПОЛНИТЕЛЬНАЯ КАЛИБРОВКА ПО КОНТРОЛЬНЫМ ТОЧКАМ

In [24]:
# Это уже не "обязательная часть", а опция.
# Она не заменяет геологическую логику, а только слегка подправляет веса
# в сторону реальных проявлений, если пользователь решил это использовать.

feature_cols = [
    "prox_facies",
    "prox_paleo",
    "prox_struct",
    "prox_magm",
    "prox_tect1",
    "prox_tect2",
    "tect_combo",
    "tect_intersection",
    "tect_magm_intersection",
    "tect_struct_intersection",
    "paleo_struct_intersection",
    "facies_paleo_intersection",
    "regional_score_sm",
    "local_score_sm",
    "geo_score",
]

grid["evidence_target"] = 0
grid["evidence_score"] = grid["geo_score"]

used_evidence_guidance = False
evidence_weights = {}

if USE_EVIDENCE_GUIDANCE and points is not None and len(points) > 0:
    try:
        joined = gpd.sjoin(
            points[["geometry"]],
            grid[["cell_id", "geometry"]],
            how="left",
            predicate="within",
        )
    except Exception:
        joined = gpd.sjoin(
            points[["geometry"]],
            grid[["cell_id", "geometry"]],
            how="left",
            op="within",
        )

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    grid.loc[grid["cell_id"].isin(positive_cells), "evidence_target"] = 1

    positives = int(grid["evidence_target"].sum())
    negatives = int((grid["evidence_target"] == 0).sum())

    if positives >= 10 and negatives > positives:
        X = grid[feature_cols].fillna(0).to_numpy()
        y = grid["evidence_target"].to_numpy()

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        lr = LogisticRegression(
            max_iter=4000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
        lr.fit(X_scaled, y)

        grid["evidence_score"] = normalize_01(lr.predict_proba(X_scaled)[:, 1])
        used_evidence_guidance = True
        evidence_weights = dict(zip(feature_cols, lr.coef_[0].tolist()))

## ИТОГ

In [25]:
grid["prospectivity_raw"] = (
    W_GEO * grid["geo_score"] +
    W_LOCAL * grid["local_score_sm"] +
    W_EVID * grid["evidence_score"]
)

grid["prospectivity"] = normalize_01(grid["prospectivity_raw"])

# Для визуальной логики, как в примере:
# меньшее значение = более перспективная ячейка
grid["prognoz"] = 1.0 - grid["prospectivity"]

top_threshold = float(grid["prospectivity"].quantile(0.90))
grid["top10"] = (grid["prospectivity"] >= top_threshold).astype(int)

# Более человекочитаемые классы
try:
    grid["prospect_class"] = pd.qcut(
        grid["prospectivity"],
        q=5,
        labels=["very_low", "low", "medium", "high", "very_high"],
        duplicates="drop",
    )
except Exception:
    grid["prospect_class"] = "medium"

## МЕТРИКИ

In [26]:
metrics = {
    "base_dir": str(BASE_DIR),
    "cell_size": CELL_SIZE,
    "grid_cells": int(len(grid)),
    "use_evidence_guidance_requested": bool(USE_EVIDENCE_GUIDANCE),
    "use_evidence_guidance_applied": bool(used_evidence_guidance),
    "top10_threshold": top_threshold,
    "point_count": int(len(points)) if points is not None else 0,
}

if points is not None and len(points) > 0:
    try:
        eval_join = gpd.sjoin(
            points[["geometry"]],
            grid[["cell_id", "prospectivity", "prognoz", "top10", "geometry"]],
            how="left",
            predicate="within",
        )
    except Exception:
        eval_join = gpd.sjoin(
            points[["geometry"]],
            grid[["cell_id", "prospectivity", "prognoz", "top10", "geometry"]],
            how="left",
            op="within",
        )

    eval_join = eval_join.dropna(subset=["cell_id"]).copy()

    if len(eval_join) > 0:
        metrics.update({
            "points_inside_grid": int(len(eval_join)),
            "mean_prospectivity_all_cells": float(grid["prospectivity"].mean()),
            "mean_prospectivity_at_points": float(eval_join["prospectivity"].mean()),
            "top10_hit_rate_at_points": float(eval_join["top10"].mean()),
            "top20_hit_rate_at_points": float(
                (eval_join["prospectivity"] >= grid["prospectivity"].quantile(0.80)).mean()
            ),
        })

if evidence_weights:
    metrics["evidence_feature_weights"] = evidence_weights

## СОХРАНЕНИЕ

In [27]:
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")

if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

top_grid = grid[grid["top10"] == 1].copy()
if len(top_grid) > 0:
    top_grid.to_file(OUT_GPKG, layer="top10_zones", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

plot_prox_magm(grid, mask, OUT_PROX_MAGM)
plot_final_map(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

OUT_METRICS.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Готово.")
print(f"Результаты сохранены в: {OUT_DIR}")
print(f"GPKG: {OUT_GPKG}")
print(f"PNG: {OUT_PNG}")
print(f"METRICS: {OUT_METRICS}")

Готово.
Результаты сохранены в: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\fixed_result
GPKG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\fixed_result\gold_forecast_fixed.gpkg
PNG: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\fixed_result\gold_forecast_fixed.png
METRICS: C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз\fixed_result\metrics.json
